# UE22CS342AA2: Data Analytics - Worksheet 3b
# ARIMAX, SARIMAX and LSTMs

Designed by Anshul Ranjan, Dept.of CSE - itsanshulranjan@gmail.com

## Student Details

• Name:

• SRN:

• Section:

In the previous worksheet, we experimented with ARIMA models. However, one caveat of ARIMA (or similar models), is that it takes only the target variable into consideration, according to the timestamp. In essence, it derives the relationship between the current target variable values and the past variable values.

However, what if we have some other external factors affecting the target values?
This is where *ARIMAX* (AutoRegressive Integrated Moving Average with eXogenous variables) steps in!

ARIMAX extends the capabilities of ARIMA by incorporating external factors or exogenous variables that influence the time series data. It's the bridge that connects the simplicity of ARIMA with the complexity of real-world forecasting, allowing us to tackle more intricate and realistic forecasting challenges.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import MinMaxScaler
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

import warnings

warnings.filterwarnings("ignore")

## Loading the data

The data is already divided into Test and Train Set!!! But always remember Before you try to forecast anything split the dataset into train-test sets, as discussed in the previous worksheet. Remember, since we're dealing with time series data, we will not perform a random split! 

In [ ]:
train = pd.read_csv("/kaggle/input/ist-traffic-and-weather-time-series-dataset/Train.csv", encoding = "unicode_escape")
test = pd.read_csv("/kaggle/input/ist-traffic-and-weather-time-series-dataset/Test.csv", encoding = "unicode_escape")

In [ ]:
train.info()

In [ ]:
# Checking for NULL values in the Data
train.isnull().sum()

In [ ]:
train.describe()

In [ ]:
train.head(10)

### Data Dictionary:

* **date_time**: Date, time, and hour of the data collected in local IST time.
* **is_holiday**: Categorical indicator of Indian national and regional holidays.
* **air_pollution_index**: Air Quality Index (range: 10-300).
* **humidity**: Numeric humidity in Celsius.
* **wind_speed**: Numeric wind speed in miles per hour.
* **wind_direction**: Cardinal wind direction (0-360 degrees).
* **visibility_in_miles**: Visibility distance in miles.
* **dew_point**: Numeric dew point in Celsius.
* **temperature**: Numeric average temperature in Kelvin.
* **rain_p_h**: Numeric amount of rain in millimeters that occurred during the hour.
* **snow_p_h**: Numeric amount of snow in millimeters that occurred during the hour.
* **clouds_all**: Numeric percentage of cloud cover.
* **weather_type**: Categorical short description of the current weather.
* **weather_description**: Categorical longer description of the current weather conditions.
* **traffic_volume**: Numeric hourly traffic volume bound in a specific direction.

We intend to perform traffic volume forecasting for each hour based on the provided time series data. Given the nature of traffic patterns, it is likely that certain times of the day, days of the week, and possibly even specific months may exhibit higher traffic volumes compared to others. This variation can often be attributed to factors such as holidays, weather conditions, and other exogenous variables.

To accurately forecast the traffic_volume attribute, we will leverage time series methods, specifically the SARIMAX (Seasonal Autoregressive Integrated Moving Average with eXogenous variables) model. This model is well-suited for capturing seasonal patterns and incorporating the influence of external factors such as air quality, weather conditions, and holidays, which can affect traffic flow. By accounting for these variables, we aim to produce precise forecasts that reflect the complex interplay of time-dependent and external influences on traffic volume.

Remember, we are performing time series analysis here. A general rule of thumb is to have the `time` column as the index column!

This line sets the  column date_time as the index of the DataFrame, allowing for time-based indexing and operations. It helps in organizing the data by date and is useful for time series analysis.

In [ ]:
train.index = train.date_time
train = train.drop(['date_time'],axis=1)
train.head()

This plot will help visualize trends in average daily traffic_volume over time, showing if traffic_volume are increasing, decreasing, or following any patterns across dates.

In [ ]:
train['traffic_volume'].plot(figsize=(25,5))

## Data Preprocessing

For handling categorical variables **is_holiday, weather_type, weather_description,** we perform one-hot encoding

In [ ]:
from tqdm import tqdm

def pre_process(data):
    data['holiday'] = 0
    for i in tqdm(range(len(data))):
        if(data.iloc[i]['is_holiday'] != "None"):
            data.iloc[i]['holiday'] = 1
    weather_type = pd.get_dummies(data['weather_type'],prefix="weather_type")
    weather_desc = pd.get_dummies(data['weather_description'],prefix="weather_desc")
    data = data.drop(['weather_type','weather_description','is_holiday'],axis=1)
    data = pd.concat([data,weather_type,weather_desc],axis=1)
    data.head()
    return(data)

In [ ]:
train = pre_process(train)

## Augmented Dickey-Fuller Test

In [ ]:
t = sm.tsa.adfuller(train.traffic_volume, autolag='AIC')
pd.Series(t[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])

The results suggest that the time series data you analyzed is **likely stationary**, meaning it does not have a unit root and its statistical properties do not change over time.

## Decomposition of elements

In [ ]:
s = sm.tsa.seasonal_decompose(train.traffic_volume, period=12)

In [ ]:
s.seasonal.plot(figsize=(20,5))

In [ ]:
s.trend.plot(figsize=(20,5))

In [ ]:
s.resid.plot(figsize=(20,5))

## ARIMAX Model

In [ ]:
train.columns

In [ ]:
plot_acf(train.traffic_volume,lags=10)
plt.show()

In [ ]:
plot_pacf(train.traffic_volume,lags=50)
plt.show()

Here, we define an algorithm which takes in a range of values of p, d, q and calculates the AIC metric on a vanilla ARIMA model. 

The **Akaike Information Criterion (AIC)** is a statistical measure used for model selection and comparison in the context of regression analysis and time series modeling.

AIC quantifies the trade-off between a model's goodness of fit and its complexity, penalizing models with too many parameters. It is employed to choose the best-fitting model among a set of candidate models. The model with the lowest AIC value is typically preferred because it represents a good balance between explaining the data and avoiding overfitting.

### Please dont run this code just for reference 

In [ ]:
'''
#Finding the best value for ARIMA
import warnings
warnings.filterwarnings("ignore")

import itertools 
p=q=range (0,7)
d = range(0,2)
pdq = list(itertools.product (p, d, q))

store = {}
for param in pdq:
       try:
              model_arima = sm.tsa.arima.ARIMA (train.traffic_volume, order = param)
              model_arima_fit = model_arima.fit()
              store[param] =  model_arima_fit.aic  
              #print(param, model_arima_fit.aic)
       except:
              continue
          
sorted_dict = dict(sorted(store.items(), key=lambda item: item[1]))
print(sorted_dict)
# The Akaike information criterion (AIC) is an estimator of in-sample prediction error and thereby relative quality of
# statistical models for a given set of data
# It's like the mean squared error in Regression - The smaller the number, the better
'''

### Assume  **p = 1 , d = 0, q = 1**

## Specify endogenous and exogenous variables in the data

In [ ]:
# Drop the 'traffic_volume' column from the training data to create the exogenous variables dataset
exog_columns = [
    'air_pollution_index',
    'humidity',
    'wind_speed',
    'wind_direction',
    'visibility_in_miles',
    'dew_point',
    'temperature',
    'rain_p_h',
    'snow_p_h',
    'clouds_all'
]

# Select only these columns from the DataFrame
exog_data = train[exog_columns]


# Add a constant term (intercept) to the exogenous variables dataset
exog = sm.add_constant(exog_data)

# Select the 'traffic_volume' column from the training data as the endogenous variable (target)
endog = train['traffic_volume']

In [ ]:
exog.dtypes

In [ ]:
mod = sm.tsa.statespace.SARIMAX(endog=endog, exog=exog, order=(1,0,1))
model_fit = mod.fit()
model_fit.summary()

# Can use model_fit = mod.fit(maxiter=5) or model_fit = mod.fit(maxiter=10) if its taking too much time to run

Question 1: Based on the provided model summary, determine the significance of each variable in the regression model. Use the p-values to classify each variable as statistically significant or not significant. Provide a brief explanation for each classification. \
Bonus if you can remove useless variables and improve the model.

In [ ]:
# Your answer here

Plotting the predicted values on the train set - shows a decent prediction

In [ ]:
train['traffic_volume'].plot(figsize=(25,10))
model_fit.fittedvalues.plot()
plt.show()

With this piece of code, we shall perform model inference. 
We'll use our hold-out test set for this. Using the exogenous variables, we'll provide input into our fitted ARIMAX model, and obtain the predcitions for `traffic_volume`

In [ ]:
predict = model_fit.predict(start = len(train),end = len(train)+len(test)-1,exog = sm.add_constant(test[[
    'air_pollution_index',
    'humidity',
    'wind_speed',
    'wind_direction',
    'visibility_in_miles',
    'dew_point',
    'temperature',
    'rain_p_h',
    'snow_p_h',
    'clouds_all'
]]))
test['predicted'] = predict.values
test.tail(5)

We've defined 2 metrics here - MAE and MAPE, to quantify our loss here. Can't calculate because the test data doesnt contain `traffic_volume` column

This how we will calculate it: 
* 
MAE = mean_absolute_error(test["traffic_volume"], test["predicted"]) \
RMSE = math.sqrt(mean_squared_error(test["traffic_volume"], test["predicted"])) \
print("MAE:", MAE) \
print("RMSE:", RMSE) 

 Question 2: Your task is to use the above learnings, and apply a SARIMAX model. Do reuse the code, identify a suitable seasonal order, and experiment to find the best performing model! Also, provide your reasoning for choosing your seasonal order!

> Hint: In your model definition step, you'll have to provide a `seasonal_order` parameter along with `order`. 

In [ ]:
# Your answer here

As before, a plot ACF and PACF of differenced time series can be used to find non-seasonal orders p and q. However, to find seasonal orders P and Q we need to plot ACF and PACF of the differenced time series at multiple seasonal steps.

From the graph, P and Q can be found

Reference :\
https://www.jadsmkbdatalab.nl/forecasting-with-sarimax-models/#:~:text=However%2C%20to%20find%20seasonal%20orders,series%20at%20multiple%20seasonal%20steps.&text=From%20left%20to%20right%3A,S%3D7

## LSTMs

Long Short-Term Memory (LSTM) is a type of recurrent neural network (RNN) architecture in deep learning. LSTMs are designed to address the vanishing gradient problem in traditional RNNs, allowing them to effectively capture and model long-range dependencies in sequential data. They have become a crucial tool for tasks like natural language processing, time series forecasting, and sequential pattern recognition.

LSTMs are often used to effectively model complicated time-series problems, so we'll explore this further.

For the scope of this worksheet, we'll only use the target variable and it's lags as input to the LSTM. However, you're encouraged to explore how the entire input dataset can be modeled as input to the model.

Our first task would be to convert our time-series forecasting problem, into a supervised learning problem. Any ideas on how we can achieve this?

Let's first learn the distinction between a time series, and a supervised learning problem.

A time series is a sequence of numbers that are ordered by a time index. This can be thought of as a list or column of ordered values.

A supervised learning problem comprises input patterns (X) and output patterns (y), such that an algorithm can learn how to predict the output patterns from the input patterns.

Pandas has a `shift()` function, that we can use to extract **lags** from the target variable. In essence, we want to somehow bring a X->y relation with respect to the target variable, while retaining the time component.

So, what are *lags*?

Lags refer to the practice of shifting a time series data point or variable backward in time by a certain number of time units.

If you're able to understand where this is going now....

We're essentially going to create a mapping such that : 

`var(t - 1) -> var(t)`; which resembles `X -> y` !

We can go further here, and take more lags, such as `var(t - 2), var(t - 3)`, etc.!

Incase you found this prelude a little difficult to follow, consider going through a more detailed write-up here: 

[Machine Learning Mastery's Blog on converting time series to supervised learning](https://machinelearningmastery.com/convert-time-series-supervised-learning-problem-python/)

## Applying LSTM to our dataset

In [ ]:
df = pd.read_csv("/kaggle/input/ist-traffic-and-weather-time-series-dataset/Train.csv", encoding = "unicode_escape")

In [ ]:
np.random.seed(11)
dataframe = df.loc[:,'traffic_volume']
dataset = dataframe.values
dataset = dataset.astype('float32')

For this demonstration purpose, we'll use lags of 7 days, and convert it into a supervised learning problem. 

Here's a function that's borrowed from the aforementioned blog, that helps us in converting the time series to a supervised problem - 

In [ ]:
# convert series to supervised learning
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1
    df = pd.DataFrame(data)
    cols, names = list(), list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [ ]:
new_dataset = series_to_supervised(dataset, 7,1)
new_dataset.head(3)

Just to give a taste of multivariate time-series forecasting using LSTMs - we'll use `temperature` and `wind_speed` in our input to the model as well:

In [ ]:
new_dataset['temperature'] = df.temperature.values[7:]
new_dataset['wind_speed']= df.wind_speed.values[7:]

In [ ]:
new_dataset = new_dataset.reindex(['temperature', 'wind_speed','var1(t-7)', 'var1(t-6)', 'var1(t-5)', 'var1(t-4)', 'var1(t-3)','var1(t-2)', 'var1(t-1)', 'var1(t)'], axis=1)
new_dataset = new_dataset.values

**Remember - we cannot use DataFrames, as LSTMs (and most other deep learning models) only accept tensors as input!**

In [ ]:
type(new_dataset)

We'll scale our features between 0 and 1 - this would be to help the process of gradient descent.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
new_dataset = scaler.fit_transform(new_dataset)

We'll split our dataset into train and test, as done for ARIMAX. Remember, it is still inherently a time series problem, so we will not use a random split.

In [ ]:
train_lstm = new_dataset[:(len(new_dataset)-30), :]
test_lstm = new_dataset[(len(new_dataset)-30):len(new_dataset), :]

We adjust the labels, such that `train_X` and `test_X` contain the features, and `train_Y`, `test_Y` contain the target

In [ ]:
train_X, train_y = train_lstm[:, :-1], train_lstm[:, -1]
test_X, test_y = test_lstm[:, :-1], test_lstm[:, -1]

The input to a LSTM is 3D - in this format: (samples, timesteps, features). We'll go ahead and reshape our train and tests sets as such

In [ ]:
train_X = train_X.reshape((train_X.shape[0], 1, train_X.shape[1]))
test_X = test_X.reshape((test_X.shape[0], 1, test_X.shape[1]))
print(train_X.shape, train_y.shape, test_X.shape, test_y.shape)

## LSTM Modeling!

In [ ]:
model = Sequential()
model.add(LSTM(32, input_shape=(train_X.shape[1], train_X.shape[2])))
model.add(Dense(1))
model.compile(loss='mae', optimizer='adam')
# fit network
history = model.fit(train_X, train_y, epochs=50, batch_size=72, verbose=2, shuffle=False)
# plot history
plt.plot(history.history['loss'], label='train')
plt.legend()
plt.show()

## LSTM Inferencing (Model Prediction)

Since we went through the whole charade of Scaling our values - making a prediction isn't completely straightforward.
We need to invert the scaling, in order to obtain the correct forecast value.

In [ ]:
# make a prediction
yhat = model.predict(test_X)

In [ ]:
test_X = test_X.reshape(test_X.shape[0], test_X.shape[2])

In [ ]:
inv_yhat = np.concatenate((yhat, test_X), axis=1)
inv_yhat = scaler.inverse_transform(inv_yhat)

In [ ]:
# invert scaling for actual
test_y = test_y.reshape((len(test_y), 1))
inv_y = np.concatenate((test_y, test_X), axis=1)
inv_y = scaler.inverse_transform(inv_y)

### Checking the performance of the model

In [ ]:
act = [i[0] for i in inv_y] # last element is the predicted power consumption
pred = [i[0] for i in inv_yhat] # last element is the actual power consumption

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

mae = mean_absolute_error(act, pred)
rmse = math.sqrt(mean_squared_error(act, pred))
print("MAE:", mae)
print("RMSE:", rmse)

Question 3: Can we use accuracy as a metric for this particular problem? Why or why not?

In [ ]:
# Your answer here

Question 4: When can LSTMs outperform ARIMA, ARIMAX or SARIMAX models? Is it worth the computational expense to fit an LSTM over a traditional time series model?

In [ ]:
# Your answer here

Question 5: What can you elucidate about the interpretability of ARIMA/ARIMAX vs LSTMs?

Hint: Think black-box models

In [ ]:
# Your answer here

### Congratulations on making it to the end of the worksheet! I hope you have a much better understanding of modeling the time-series workflow, and applications of Deep Learning methods too!

### Incase you want to explore further, Facebook Prophet is a great time series model as well to have in your toolbox!
### Read more [here!](https://www.kaggle.com/code/prashant111/tutorial-time-series-forecasting-with-prophet)